# Adapting EMCAD (PVT-EMCAD-B2) to 2D multi-class brain tumor segmentation on MSD BrainTumour

Pipeline: (1) setup -> (2) preprocessing -> (3) dataloader -> (4) model (4-channel input, 4 classes) -> (5) loss -> (6) training -> (7) 3D evaluation (Dice, HD95) -> (8) overlay figure.

Requires a GPU and **Internet = On** (clones the EMCAD repo and downloads the PVTv2-b2 weights).
Attach both MSD datasets (training set and validation set) before running.

## 1. Setup

In [ ]:
import os, sys, glob, time, random, subprocess, warnings
import numpy as np, pandas as pd, nibabel as nib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import distance_transform_edt, binary_erosion
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
warnings.filterwarnings("ignore")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

W = "/kaggle/working"
device = "cuda"

# EMCAD repository and original-format PVTv2-b2 pretrained weights
if not os.path.isdir(f"{W}/EMCAD/lib"):
    subprocess.run(["git", "clone", "https://github.com/SLDGroup/EMCAD.git", f"{W}/EMCAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"], check=True)
WEIGHTS = f"{W}/pvt_v2_b2.pth"
if not os.path.exists(WEIGHTS) or os.path.getsize(WEIGHTS) < 1e6:
    subprocess.run(["wget", "-q", "-O", WEIGHTS,
                    "https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth"], check=True)

open(f"{W}/EMCAD/lib/__init__.py", "a").close()
sys.path.insert(0, f"{W}/EMCAD")
print("setup OK | weights (MB):", round(os.path.getsize(WEIGHTS) / 1e6, 1))

## 2. Preprocessing
MSD volumes are `(240, 240, 155, 4)` with channel order FLAIR, T1w, T1gd (T1ce), T2w.
Each volume is z-score normalised per modality over brain voxels, cut into axial slices, resized to 128x128 and saved as `.npy`.
Subset used: first 80 training patients, first 20 labeled validation patients (patient-level split).

In [ ]:
TR = "/kaggle/input/datasets/azizesseghaier/msd-brain-tumor-mri-training-set/Brain Tumor Training Set"
VA = "/kaggle/input/datasets/azizesseghaier/msd-brain-tumor-segmentation-validation-set/Val set"
OUT = f"{W}/data"
SIZE, N_TRAIN, N_VAL = 128, 80, 20

def process(img_path, lab_path, out_path):
    x = nib.load(img_path).get_fdata().astype(np.float32)
    y = nib.load(lab_path).get_fdata().astype(np.uint8)
    ch = [i for i, s in enumerate(x.shape) if s == 4][0]    # modality axis
    x = np.moveaxis(x, ch, 0)                               # (4, H, W, D)
    x = np.transpose(x, (0, 3, 1, 2))                       # (4, D, H, W): axial slices
    y = np.transpose(y, (2, 0, 1))                          # (D, H, W)
    for c in range(4):                                      # per-modality z-score on brain voxels
        m = x[c] > 0
        if m.sum() > 0:
            x[c][m] = (x[c][m] - x[c][m].mean()) / (x[c][m].std() + 1e-8)
        x[c][~m] = 0
    xt = F.interpolate(torch.from_numpy(x).permute(1, 0, 2, 3), size=(SIZE, SIZE),
                       mode="bilinear", align_corners=False)
    yt = F.interpolate(torch.from_numpy(y)[:, None].float(), size=(SIZE, SIZE), mode="nearest")
    np.save(out_path + "_img.npy", xt.numpy().astype(np.float16))
    np.save(out_path + "_lab.npy", yt[:, 0].numpy().astype(np.uint8))

def run(img_dir, lab_dir, split, n):
    os.makedirs(f"{OUT}/{split}", exist_ok=True)
    for f in sorted(os.listdir(lab_dir))[:n]:
        base = f"{OUT}/{split}/{f.replace('.nii', '')}"
        if not os.path.exists(base + "_lab.npy"):
            process(os.path.join(img_dir, f), os.path.join(lab_dir, f), base)
    print(split, "done")

run(f"{TR}/training_images", f"{TR}/training_labels", "train", N_TRAIN)
run(f"{VA}/val images", f"{VA}/val labels", "val", N_VAL)

## 3. Dataloader
4-channel axial slices. Training keeps all tumor slices plus 20% of tumor-free brain slices; validation keeps all brain slices. Training augmentation: random horizontal/vertical flips.

In [ ]:
class BratsSlices(Dataset):
    def __init__(self, folder, train=True, keep_empty_frac=0.2, seed=0):
        self.train = train
        self.imgs, self.labs, self.index = [], [], []
        rng = np.random.RandomState(seed)
        for p in sorted(glob.glob(folder + "/*_img.npy")):
            x = np.load(p)                           # (D, 4, S, S) float16
            y = np.load(p.replace("_img", "_lab"))   # (D, S, S) uint8
            pid = len(self.imgs); self.imgs.append(x); self.labs.append(y)
            for z in range(x.shape[0]):
                if not (np.abs(x[z]) > 0).mean() > 0.02: continue      # skip slices with almost no brain
                if (y[z] > 0).any() or (not train) or rng.rand() < keep_empty_frac:
                    self.index.append((pid, z))

    def __len__(self): return len(self.index)

    def __getitem__(self, i):
        pid, z = self.index[i]
        x = torch.from_numpy(self.imgs[pid][z].astype(np.float32))    # (4, S, S)
        y = torch.from_numpy(self.labs[pid][z].astype(np.int64))      # (S, S)
        if self.train:
            if np.random.rand() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if np.random.rand() < 0.5: x, y = x.flip(-2), y.flip(-2)
        return x, y

train_ds = BratsSlices(f"{OUT}/train", train=True)
val_ds = BratsSlices(f"{OUT}/val", train=False)
print("train slices:", len(train_ds), "| val slices:", len(val_ds))

## 4. Model
PVT-EMCAD-B2 with (a) the first patch-embedding conv widened from 3 to 4 input channels (4th channel = mean of the pretrained 3) and (b) 4 output classes in all four segmentation heads.

In [ ]:
os.chdir(f"{W}/EMCAD")
from lib.networks import EMCADNet

def make_model(num_classes=4):
    m = EMCADNet(num_classes=num_classes, encoder="pvt_v2_b2", pretrain=True, pretrained_dir=W)
    ck = torch.load(WEIGHTS, map_location="cpu")
    sd = m.backbone.state_dict()
    n = sum(1 for k, v in ck.items() if k in sd and sd[k].shape == v.shape)
    print(f"pretrained keys matched: {n} of {len(sd)} backbone tensors")
    for name, mod in m.backbone.named_modules():
        if isinstance(mod, nn.Conv2d) and mod.in_channels == 3:
            new = nn.Conv2d(4, mod.out_channels, mod.kernel_size, mod.stride, mod.padding,
                            bias=mod.bias is not None)
            with torch.no_grad():
                new.weight[:, :3] = mod.weight
                new.weight[:, 3:] = mod.weight.mean(1, keepdim=True)
                if mod.bias is not None: new.bias.copy_(mod.bias)
            parent = m.backbone
            *path, last = name.split(".")
            for p in path: parent = getattr(parent, p)
            setattr(parent, last, new)
            print("replaced first conv:", name)
            break
    return m

## 5. Loss
0.7 x Dice + 0.3 x cross-entropy on each of the four decoder heads, plus one extra term on the sum of the head logits (simplified version of the paper's multi-head combination loss).

In [ ]:
def dice_loss(logits, y, eps=1e-5):
    p = logits.softmax(1)
    oh = F.one_hot(y, 4).permute(0, 3, 1, 2).float()
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    return 1 - ((2 * inter + eps) / (den + eps)).mean()

def seg_loss(logits, y):
    return 0.7 * dice_loss(logits, y) + 0.3 * F.cross_entropy(logits, y)

def total_loss(outs, y):
    return sum(seg_loss(o, y) for o in outs) + seg_loss(sum(outs), y)

## 6. Training
AdamW (lr 1e-4, wd 1e-4), cosine schedule, batch 16, mixed precision, 15 epochs. Best epoch chosen by mean slice-level Dice (WT/TC/ET) on the validation set. Prediction = argmax of the sum of the four head logits.

In [ ]:
EPOCHS, BS, LR = 15, 16, 1e-4
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

model = make_model().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

def regions(lbl):   # nested regions from label map: WT = 1,2,3 | TC = 2,3 | ET = 3
    return {"WT": lbl > 0, "TC": (lbl == 2) | (lbl == 3), "ET": lbl == 3}

@torch.no_grad()
def evaluate():
    model.eval()
    inter = {k: 0. for k in ["WT", "TC", "ET"]}; tot = {k: 0. for k in inter}
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            outs = model(x)
        pred = sum(o.float() for o in outs).argmax(1)
        rp, rg = regions(pred), regions(y)
        for k in inter:
            inter[k] += (rp[k] & rg[k]).sum().item()
            tot[k] += rp[k].sum().item() + rg[k].sum().item()
    d = {k: 2 * inter[k] / (tot[k] + 1e-8) for k in inter}
    d["mean"] = float(np.mean([d["WT"], d["TC"], d["ET"]]))
    return d

best = 0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tl = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            outs = model(x)
        loss = total_loss([o.float() for o in outs], y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item()
    sched.step()
    d = evaluate()
    print(f"ep {ep+1}/{EPOCHS} | loss {tl/len(train_loader):.3f} | "
          f"Dice WT {d['WT']:.3f} TC {d['TC']:.3f} ET {d['ET']:.3f} mean {d['mean']:.3f} | {time.time()-t0:.0f}s")
    if d["mean"] > best:
        best = d["mean"]; torch.save(model.state_dict(), f"{W}/best.pth")
        print("   saved best.pth")

## 7. Evaluation: per-patient 3D Dice and HD95
Slices of each validation patient are predicted and stacked into a 3D volume. HD95 in mm with approximate spacing (1 mm between slices, 240/128 mm in-plane after resizing).

In [ ]:
model.load_state_dict(torch.load(f"{W}/best.pth")); model.eval()
SPACING = (1.0, 240/128, 240/128)

def dice(p, g):
    if p.sum() == 0 and g.sum() == 0: return 1.0
    return 2 * (p & g).sum() / (p.sum() + g.sum())

def hd95(p, g):
    if p.sum() == 0 or g.sum() == 0: return np.nan      # undefined; counted separately
    pb, gb = p ^ binary_erosion(p), g ^ binary_erosion(g)
    d_to_g = distance_transform_edt(~gb, sampling=SPACING)
    d_to_p = distance_transform_edt(~pb, sampling=SPACING)
    return np.percentile(np.concatenate([d_to_g[pb], d_to_p[gb]]), 95)

rows, preds = [], {}
with torch.no_grad():
    for pid in range(len(val_ds.imgs)):
        vol, gt = val_ds.imgs[pid], val_ds.labs[pid]
        out = []
        for i in range(0, len(vol), 32):
            x = torch.from_numpy(vol[i:i+32].astype(np.float32)).to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                o = model(x)
            out.append(sum(t.float() for t in o).argmax(1).cpu().numpy())
        pred = np.concatenate(out).astype(np.uint8); preds[pid] = pred
        rp, rg = regions(pred), regions(gt)
        row = {"patient": pid}
        for k in ["WT", "TC", "ET"]:
            row[f"Dice_{k}"] = dice(rp[k], rg[k]); row[f"HD95_{k}"] = hd95(rp[k], rg[k])
        rows.append(row)

df = pd.DataFrame(rows)
summary = pd.DataFrame({
    "Dice": [df[f"Dice_{k}"].mean() for k in ["WT", "TC", "ET"]],
    "HD95 (mm)": [df[f"HD95_{k}"].mean() for k in ["WT", "TC", "ET"]],
    "n HD95 undefined": [df[f"HD95_{k}"].isna().sum() for k in ["WT", "TC", "ET"]],
}, index=["WT", "TC", "ET"])
summary.loc["Mean"] = [summary["Dice"].mean(), summary["HD95 (mm)"].mean(), np.nan]
print(summary.round(3))
df.to_csv(f"{W}/per_patient_metrics.csv", index=False)

## 8. Overlay figure
Best, median and worst patients (by mean 3D Dice) at their largest-tumor slice, plus the worst-performing tumor slice overall.

In [ ]:
df["mean_dice"] = df[["Dice_WT", "Dice_TC", "Dice_ET"]].mean(1)
order = df.sort_values("mean_dice").patient.tolist()
best_p, med_p, worst_p = order[-1], order[len(order)//2], order[0]

def biggest_slice(pid):
    return int(np.argmax((val_ds.labs[pid] > 0).sum((1, 2))))

fail = (None, None, 2.0)       # worst per-slice WT Dice among slices with a reasonably sized tumor
for pid in range(len(val_ds.imgs)):
    g, p = val_ds.labs[pid] > 0, preds[pid] > 0
    for z in range(g.shape[0]):
        if g[z].sum() > 150:
            d = 2 * (g[z] & p[z]).sum() / (g[z].sum() + p[z].sum() + 1e-8)
            if d < fail[2]: fail = (pid, z, d)

cases = [(best_p, biggest_slice(best_p), "Best patient"),
         (med_p, biggest_slice(med_p), "Median patient"),
         (worst_p, biggest_slice(worst_p), "Worst patient"),
         (fail[0], fail[1], f"Failure slice (WT Dice {fail[2]:.2f})")]

cmap = ListedColormap([(0, 0, 0, 0), (0, 1, 0, 0.55), (0, 0.4, 1, 0.55), (1, 0, 0, 0.6)])
cols = ["FLAIR", "T1ce", "Ground truth", "Prediction"]
fig, ax = plt.subplots(len(cases), 4, figsize=(12, 3 * len(cases)))
for r, (pid, z, title) in enumerate(cases):
    x = val_ds.imgs[pid][z].astype(np.float32); g = val_ds.labs[pid][z]; p = preds[pid][z]
    flair, t1ce = x[0], x[2]
    ax[r, 0].imshow(flair, cmap="gray"); ax[r, 1].imshow(t1ce, cmap="gray")
    ax[r, 2].imshow(flair, cmap="gray"); ax[r, 2].imshow(g, cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    ax[r, 3].imshow(flair, cmap="gray"); ax[r, 3].imshow(p, cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    for c in range(4):
        ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(cols[c])
    ax[r, 0].text(-0.05, 0.5, f"{title}\n(val patient {pid}, slice {z})", transform=ax[r, 0].transAxes,
                  ha="right", va="center", fontsize=9)
fig.legend(handles=[Patch(color=(0, 1, 0), label="Edema (1)"), Patch(color=(0, 0.4, 1), label="Non-enhancing (2)"),
                    Patch(color=(1, 0, 0), label="Enhancing (3)")], loc="lower center", ncol=3)
plt.tight_layout(rect=[0.08, 0.04, 1, 1])
plt.savefig(f"{W}/overlays.png", dpi=150, bbox_inches="tight"); plt.show()